# ICS2123 - Tarea 4 Código

### Celda 1: Importamos algunas librerías (NO MODIFICAR)

In [1]:
# Celda 1
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import optuna
from collections import defaultdict
from bisect import insort

c:\Users\fcota\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Inciso c)

### Celda 2: Parámetros Iniciales (NO MODIFICAR)

In [2]:
# Celda 2
lambd = 3/4 # Tasa de llegada

mu1 = 1/5 # Tasa de servicio del servidor 1
mu2 = 1/6 # Tasa de servicio del servidor 2
mu3 = 1/3 # Tasa de servicio del servidor 3
mu4 = 1/4 # Tasa de servicio del servidor 4

C1 = 6 # Capacidad del servidor 1
C2 = 3 # Capacidad del servidor 2
C3 = 4 # Capacidad del servidor 3
C4 = 2 # Capacidad del servidor 4

p1 = 0.3 # Proporción de clientes que van del servidor 1 al 2
p2 = 0.2 # Proporción de clientes que van del servidor 1 al 3
p3 = 0.1 # Proporción de clientes que van del servidor 3 al 2
p4 = 0.4 # Proporción de clientes que salen del sistema insatisfactoriamente

num_simulaciones_largas = 100 # Número de simulaciones (para el inciso (c) y (d))
num_minutos_simulaciones_largas = 5000* 60 # 5000 minutos (para el inciso (c) y (d))

num_simulaciones_cortas = 10000 # Número de simulaciones cortas (para el inciso (f))
num_minutos_simulaciones_cortas = 1*60 # 1 horas (para el inciso (f))


### Celda 3: Clase Cliente (NO MODIFICAR)

In [3]:
# Celda 3
class Cliente:
    def __init__(self, id, tiempo_entrada):
        self.id = id
        self.tiempo_1 = tiempo_entrada
        self.tiempo_2 = tiempo_entrada
        self.tiempo_3 = tiempo_entrada
        self.tiempo_4 = tiempo_entrada
        self.cola_1 = False
        self.cola_2 = False
        self.cola_3 = False
        self.cola_4 = False

### Celda 4: Simulación (MODIFICAR)

In [4]:
# Celda 4
def simular_estaciones(
    l, mu1, mu2, mu3, mu4,
    C1, C2, C3, C4,
    p1, p2, p3, p4, T_simulacion, seed):
    
    np.random.seed(seed)

    # Variables globales
    t = 0
    tiempo_anterior = 0

    servidores1 = 0
    servidores2 = 0
    servidores3 = 0
    servidores4 = 0

    clientes = {}
    clientes_cola_1 = []
    clientes_cola_2 = []
    clientes_cola_3 = []
    clientes_cola_4 = []
    clientes_estaciones = {}
    clientes_insatisfechos = {}

    id = 0
    id_1 = 0
    id_2 = 0
    id_3 = 0
    id_4 = 0

    id_cola = 0
    id_cola_1 = 0
    id_cola_2 = 0
    id_cola_3 = 0
    id_cola_4 = 0

    estado_1 = 0
    estado_2 = 0
    estado_3 = 0
    estado_4 = 0

    lista_eventos = []


    def agregar_evento(tipo, tiempo, id):
        insort(lista_eventos, [tiempo, tipo, id])

    agregar_evento('llegada', np.random.exponential(1/l), id) # Recordar que numpy utiliza la media como parámetro de la exponencial
    id += 1

    while t < T_simulacion or len(lista_eventos) > 0:
        tiempo_anterior = t
        t, evento, id_selecionado = lista_eventos.pop(0)

        if evento == 'llegada':
            estado_1 += 1
            if servidores1 < C1:
                servidores1 += 1
                nuevo_cliente = Cliente(id_selecionado, t)
                tiempo_servicio = np.random.exponential(1/mu1) 
                agregar_evento('salida_1', t + tiempo_servicio, id_selecionado)
                id_1 += 1
            else:
                nuevo_cliente = Cliente(id_selecionado, t)
                nuevo_cliente.cola_1 = True
                insort(clientes_cola_1, (t, id_selecionado))
                id_cola_1 += 1
                id_cola += 1
            clientes[id_selecionado] = nuevo_cliente
            if t < T_simulacion:
                agregar_evento('llegada', np.random.exponential(1/l) + t, id)
                id += 1

        elif evento == 'salida_1':
            estado_1 -= 1
            servidores1 -= 1
            p = np.random.uniform(0, 1)
            if p <= p1:
                estado_2 += 1
                if servidores2 < C2:
                    servidores2 += 1
                    tiempo_servicio = np.random.exponential(1/mu2)
                    agregar_evento('salida_2', t + tiempo_servicio, id_selecionado)
                    id_2 += 1
                else:
                    cliente = clientes[id_selecionado]
                    cliente.cola_2 = True
                    cliente.tiempo_2 = t
                    insort(clientes_cola_2, (t, id_selecionado))
                    id_cola_2 += 1
                    id_cola += 1

            elif p <= p1 + p2:
                estado_3 += 1
                if servidores3 < C3:
                    servidores3 += 1
                    tiempo_servicio = np.random.exponential(1/mu3)
                    agregar_evento('salida_3', t + tiempo_servicio, id_selecionado)
                    id_3 += 1
                else:
                    cliente = clientes[id_selecionado]
                    cliente.cola_3 = True
                    cliente.tiempo_3 = t
                    insort(clientes_cola_3, (t, id_selecionado))
                    id_cola_3 += 1
                    id_cola += 1

            else:
                cliente = clientes[id_selecionado]
                clientes_estaciones[id_selecionado] = cliente
                del clientes[id_selecionado]

            if clientes_cola_1:
                servidores1 += 1
                tiempo_1, cliente_id = clientes_cola_1.pop(0)
                cliente = clientes[cliente_id]
                tiempo_servicio = np.random.exponential(1/mu1)
                agregar_evento('salida_1', t + tiempo_servicio, cliente_id)
                id_1 += 1

        elif evento == 'salida_2':
            estado_2 -= 1
            servidores2 -= 1
            estado_4 += 1
            if servidores4 < C4:
                servidores4 += 1
                tiempo_servicio = np.random.exponential(1/mu4)
                agregar_evento('salida_4', t + tiempo_servicio, id_selecionado)
                id_4 += 1
            else:
                cliente = clientes[id_selecionado]
                cliente.cola_4 = True
                cliente.tiempo_4 = t
                insort(clientes_cola_4, (t, id_selecionado))
                id_cola_4 += 1
                id_cola += 1

            if clientes_cola_2:
                servidores2 += 1
                tiempo_2, cliente_id = clientes_cola_2.pop(0)
                cliente = clientes[cliente_id]
                tiempo_servicio = np.random.exponential(1/mu2)
                agregar_evento('salida_2', t + tiempo_servicio, cliente_id)
                id_2 += 1


        elif evento == 'salida_3':
            estado_3 -= 1
            servidores3 -= 1
            p = np.random.uniform(0, 1)
            if p <= p3:
                estado_2 += 1
                if servidores2 < C2:
                    servidores2 += 1
                    tiempo_servicio = np.random.exponential(1/mu2)
                    agregar_evento('salida_2', t + tiempo_servicio, id_selecionado)
                    id_2 += 1
                else:
                    cliente = clientes[id_selecionado]
                    cliente.cola_2 = True
                    cliente.tiempo_2 = t
                    insort(clientes_cola_2, (t, id_selecionado))
                    id_cola_2 += 1
                    id_cola += 1

            else:
                estado_4 += 1
                if servidores4 < C4:
                    servidores4 += 1
                    tiempo_servicio = np.random.exponential(1/mu4)
                    agregar_evento('salida_4', t + tiempo_servicio, id_selecionado)
                    id_4 += 1
                else:
                    cliente = clientes[id_selecionado]
                    cliente.cola_4 = True
                    cliente.tiempo_4 = t
                    insort(clientes_cola_4, (t, id_selecionado))
                    id_cola_4 += 1
                    id_cola += 1

            if clientes_cola_3:
                servidores3 += 1
                tiempo_3, cliente_id = clientes_cola_3.pop(0)
                cliente = clientes[cliente_id]
                tiempo_servicio = np.random.exponential(1/mu3)
                agregar_evento('salida_3', t + tiempo_servicio, cliente_id)
                id_3 += 1

        elif evento == 'salida_4':
            estado_4 -= 1
            servidores4 -= 1
            p = np.random.uniform(0, 1)
            cliente = clientes[id_selecionado]
            if p <= p4:
                clientes_insatisfechos[id_selecionado] = cliente
            else:
                clientes_estaciones[id_selecionado] = cliente
            del clientes[id_selecionado]

            if clientes_cola_4:
                servidores4 += 1
                tiempo_4, cliente_id = clientes_cola_4.pop(0)
                cliente = clientes[cliente_id]
                tiempo_servicio = np.random.exponential(1/mu4)
                agregar_evento('salida_4', t + tiempo_servicio, cliente_id)
                id_4 += 1

    return "Simulacion Lista"


### Celda 5: Ejecutar Simulación

In [5]:
# Celda 5
simular_estaciones(
    lambd, mu1, mu2, mu3, mu4,
    C1, C2, C3, C4,
    p1, p2, p3, p4,
    num_minutos_simulaciones_largas, 2123
)

'Simulacion Lista'

## Inciso d)

## Inciso e)

### Celda 6: Parámetros para medir costos (NO MODIFICAR)

In [6]:
# Celda 6
S_1 = [9000, 10000, 8000, 7000] # Costos del tipo 1
S_2 = [30000, 50000, 70000, 20000] # Costos del tipo 2
S_3 = [5000, 15000, 10000, 5000] # Costos del tipo 3
S_4 = [80000, 100000, 60000, 50000] # Costos del tipo 4

alpha_1 = 0.1 # Proporción costos del tipo 1
alpha_2 = 0.4 # Proporción costos del tipo 2
alpha_3 = 0.3 # Proporción costos del tipo 3
alpha_4 = 0.2 # Proporción costos del tipo 4

## Inciso f)